# fase_2 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 2.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()

# Mengambil nama tabel dari hasil query
# Note: Format output 'SHOW TABLES' biasanya {'Tables_in_dbname': 'tablename'}
target_tables = [list(t.values())[0] for t in tables_data]

print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")
print(target_tables)

# Dictionary untuk menyimpan data yang sudah di-load
df_old = {}

print("\n--- Memulai proses load semua data tabel ---")

for table in target_tables:
    try:
        # Load data menggunakan pandas langsung dari koneksi SQL
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")
print("Kamu sekarang bisa akses datanya dengan: df_old['nama_tabel']")

# Contoh akses data:
# print(df_old['users'].head())

# Tutup koneksi jika sudah tidak digunakan
# db_old.close()
# db_new.close()


--- Ditemukan 108 tabel di Database Lama ---
['absensi', 'absensi_note', 'bidang', 'bidangkategori', 'bidanglink', 'calon', 'calon_detil', 'calon_pertanyaan', 'calon_pertanyaan_detil', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_siswa', 'catatan_siswa_follow_up', 'catatanawal_admin', 'catatanawal_datautama', 'catatanawal_infolain', 'catatanawal_tglpenting', 'divisi', 'docs', 'file_rapor_siswa', 'form', 'form_calon', 'form_calon_detil1', 'form_calon_detil2', 'form_calon_detil3', 'form_calon_detil4', 'format_rapor', 'format_rapor_detil', 'format_rapor_detil_rumus', 'format_rapor_rumus', 'format_raport_level', 'hakakses', 'histori_pengajuan', 'history_rapor', 'identitas', 'infrastruktur', 'jabatan', 'jadwal', 'jadwal_detil', 'jadwal_pengajar', 'jadwal_siswa', 'jamkerja', 'kabupaten', 'karyawan', 'kecamatan', 'keluar', 'keluarga', 'kelurahan', 'kurikulum', 'kurikulum_detil', 'kurikulum_detil_sub', 'kurikulum_kelas', 'kursus', 'leapprofil', 'leapverse', 'level', 'lib

In [4]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS (DATABASE BARU)
# ---------------------------------------------------------
# Menggunakan cursor dari database baru
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()

# Mengambil nama tabel dari hasil query
target_tables_new = [list(t.values())[0] for t in tables_data_new]

print(f"\n--- Ditemukan {len(target_tables_new)} tabel di Database Baru ---")
print(target_tables_new)

# Dictionary untuk menyimpan data dari database baru (jika diperlukan untuk verifikasi)
df_new = {}

print("\n--- Memulai proses load semua data dari Database Baru ---")

for table in target_tables_new:
    try:
        # Load data menggunakan pandas dengan koneksi database baru
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table} dari DB Baru: {e}")

print("\n--- Proses load selesai. Data DB Baru tersimpan di 'df_new' ---")


--- Ditemukan 104 tabel di Database Baru ---
['absensi', 'activity_log', 'admin_sarpras', 'bidang_kategori', 'bidang_link', 'busdev_bidang', 'cache', 'cache_locks', 'calon_siswa', 'calon_siswa_akademik', 'calon_siswa_bayar', 'calon_siswa_jadwal', 'calon_siswa_kursus', 'calon_siswa_ortu', 'calon_siswa_proses', 'calon_siswa_status_logs', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_siswa', 'division_user', 'divisions', 'failed_jobs', 'followup_cs', 'histori_pengajuan', 'izin_karyawan', 'jadwal', 'jadwal_detail', 'jadwal_detail_logs', 'jadwal_hari', 'jadwal_pengajar', 'jadwal_siswa', 'job_batches', 'jobs', 'kabupaten', 'karyawan', 'karyawan_resign', 'kecamatan', 'keluarga_karyawan', 'kelurahan', 'kemitraan_verifikator', 'kontak_prospek', 'kursus', 'kursus_level', 'kursus_libur', 'kursus_siswa', 'level', 'libur', 'log_aktivitas', 'migrations', 'mitra', 'mitra_progres', 'model_has_permissions', 'model_has_roles', 'mou', 'parameter_nilai', 'password_reset_tokens', 'pel

## 3. Transform Data (jika diperlukan)

karyawan, keluarga_karyawan, bidang_kategori, bidang_link.

In [5]:
cols_to_keep_from_old = [col for col in df_old['users'].columns 
                        if col not in ['nama', 'pass'] 
                        and col not in df_new['users'].columns]

# Merge berdasarkan email
df_merged_users = df_new['users'].merge(
    df_old['users'][['email'] + cols_to_keep_from_old],
    on='email',
    how='left'
)

display(df_merged_users.info())

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_user            51 non-null     str    
 1   name               51 non-null     str    
 2   email              51 non-null     str    
 3   email_verified_at  0 non-null      object 
 4   password           51 non-null     str    
 5   remember_token     0 non-null      object 
 6   created_at         0 non-null      object 
 7   updated_at         0 non-null      object 
 8   idusers            51 non-null     str    
 9   foto               44 non-null     str    
 10  idrole             51 non-null     str    
 11  wa                 50 non-null     str    
 12  thnbekerja         51 non-null     str    
 13  idjabatan          51 non-null     str    
 14  idjamkerja         14 non-null     float64
 15  minat              50 non-null     str    
 16  status             51 non-null     str 

None

In [6]:
# Create a new dictionary with the updated 'idusers' values
df_new_users_dict = df_merged_users.set_index('idusers')['id_user'].to_dict()

# Replace the 'idusers' values in df_old['karyawan'] with the corresponding values from df_new_users_dict
df_old['karyawan']['idusers'] = df_old['karyawan']['idusers'].map(df_new_users_dict)

# Display semua baris dan kolom
# print("\n5. TAMPILAN SEMUA DATA:")
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', None)
# pd.set_option('display.max_colwidth', None)
# display(df_old['karyawan'])

In [7]:
# Create a new dictionary with the updated 'idusers' values
df_new_users_dict = df_merged_users.set_index('id_user')['thnbekerja'].to_dict()

# Replace the 'idusers' values in df_old['karyawan'] with the corresponding values from df_new_users_dict
df_old['karyawan']['thnbekerja'] = df_old['karyawan']['idusers'].map(df_new_users_dict)

# # Display semua baris dan kolom
# print("\n5. TAMPILAN SEMUA DATA:")
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', None)
# pd.set_option('display.max_colwidth', None)
# display(df_old['karyawan'])

In [8]:
# Display only the 'id_karyawan' and 'nama_karyawan' columns
print("\n5. TAMPILAN SEMUA DATA:")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df_old['karyawan'][['idkaryawan', 'nama', 'thnbekerja']])


5. TAMPILAN SEMUA DATA:


,idkaryawan,nama,thnbekerja
0,LEAP001VI02,NaN,2023-06-02
1,LEAP003III2023,Graciela Evanda Ronadi,2023-03-13
2,LEAP011XII01,DANIAR AULIA RIZKI,2020-12-01
3,LEAP012III02,Habibah Melyna Elfiani,2020-03-02
4,LEAP014VII31,Laksmi Puspitowardhani,2018-07-31
5,LEAP015II2011,NaN,14-2-2011
6,LEAP016VI2017,Luluk Fatikah Sari,2017-06-19
7,LEAP018IV20,Ditari Kurnia Damayanti,2021-04-20
8,LEAP019VI2019,Hartatik,2019-06-01
9,LEAP020IV30,Juni Arlianto,2009-04-30


In [11]:
import pandas as pd

# Membuat kolom baru untuk konversi bulan menjadi angka romawi
df_old['karyawan']['bulan_romawi'] = df_old['karyawan']['thnbekerja'].apply(lambda x: pd.Period(x).strftime('%B').upper())

# Merubah setiap bulan menjadi angka romawi
bulan_romawi_mapping = {
    'JANUARY': 'I',
    'FEBRUARY': 'II',
    'MARCH': 'III',
    'APRIL': 'IV',
    'MAY': 'V',
    'JUNE': 'VI',
    'JULY': 'VII',
    'AUGUST': 'VIII',
    'SEPTEMBER': 'IX',
    'OCTOBER': 'X',
    'NOVEMBER': 'XI',
    'DECEMBER': 'XII'
}

df_old['karyawan']['bulan_romawi'] = df_old['karyawan']['bulan_romawi'].replace(bulan_romawi_mapping)

# Membuat kolom baru untuk tanggal dan tahun
df_old['karyawan']['tanggal'] = df_old['karyawan']['thnbekerja'].apply(lambda x: pd.Period(x).strftime('%d'))
df_old['karyawan']['tahun'] = df_old['karyawan']['thnbekerja'].apply(lambda x: pd.Period(x).year)

# Update df_old['karyawan']['idkaryawan'] dengan struktur penamaan yang diinginkan

df_old['karyawan']['idkaryawan'] = df_old['karyawan']['idkaryawan'].str[:7] + df_old['karyawan']['tanggal'] + df_old['karyawan']['bulan_romawi'] + df_old['karyawan']['tahun'].astype(str).str[-2:]

# Display only the 'id_karyawan' and 'nama_karyawan' columns
print("\n5. TAMPILAN SEMUA DATA:")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df_old['karyawan'][['idkaryawan', 'nama', 'thnbekerja']])


5. TAMPILAN SEMUA DATA:


,idkaryawan,nama,thnbekerja
0,LEAP00102VI23,NaN,2023-06-02
1,LEAP00313III23,Graciela Evanda Ronadi,2023-03-13
2,LEAP01101XII20,DANIAR AULIA RIZKI,2020-12-01
3,LEAP01202III20,Habibah Melyna Elfiani,2020-03-02
4,LEAP01431VII18,Laksmi Puspitowardhani,2018-07-31
5,LEAP01514II11,NaN,14-2-2011
6,LEAP01619VI17,Luluk Fatikah Sari,2017-06-19
7,LEAP01820IV21,Ditari Kurnia Damayanti,2021-04-20
8,LEAP01901VI19,Hartatik,2019-06-01
9,LEAP02030IV09,Juni Arlianto,2009-04-30



5. TAMPILAN SEMUA DATA:


,idkaryawan,nama,thnbekerja
0,LEAP00102VI23,NaN,2023-06-02
1,LEAP00313III23,Graciela Evanda Ronadi,2023-03-13
2,LEAP01101XII20,DANIAR AULIA RIZKI,2020-12-01
3,LEAP01202III20,Habibah Melyna Elfiani,2020-03-02
4,LEAP01431VII18,Laksmi Puspitowardhani,2018-07-31
5,LEAP01514II11,NaN,14-2-2011
6,LEAP01619VI17,Luluk Fatikah Sari,2017-06-19
7,LEAP01820IV21,Ditari Kurnia Damayanti,2021-04-20
8,LEAP01901VI19,Hartatik,2019-06-01
9,LEAP02030IV09,Juni Arlianto,2009-04-30


In [ ]:
# Analisis df_merged_users untuk duplicate columns, null values, dan informasi lainnya

print("="*80)
print("ANALISIS df_merged_users")
print("="*80)

# 1. Info umum dataframe
print("\n1. INFORMASI UMUM:")
print(f"   Shape: {df_merged_users.shape}")
print(f"   Total Rows: {len(df_merged_users)}")
print(f"   Total Columns: {len(df_merged_users.columns)}")

# 2. Cek duplicate columns
print("\n2. DUPLICATE COLUMNS:")
duplicate_cols = df_merged_users.columns[df_merged_users.columns.duplicated()]
if len(duplicate_cols) > 0:
    print(f"   ⚠️  Ditemukan {len(duplicate_cols)} duplicate column(s):")
    for col in duplicate_cols.unique():
        print(f"      - {col}")
else:
    print("   ✓ Tidak ada duplicate columns")

# 3. Cek null values per column
print("\n3. NULL VALUES PER COLUMN:")
null_counts = df_merged_users.isnull().sum()
null_percentage = (null_counts / len(df_merged_users)) * 100
null_info = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Null_Percentage': null_percentage.values
}).sort_values('Null_Count', ascending=False)

null_info_display = null_info[null_info['Null_Count'] > 0]
if len(null_info_display) > 0:
    print(null_info_display.to_string(index=False))
else:
    print("   ✓ Tidak ada null values")

# 4. Data types
print("\n4. DATA TYPES:")
# Display data types with a more detailed summary
dtype_summary = pd.DataFrame({
    'Column': df_merged_users.dtypes.index,
    'Data_Type': df_merged_users.dtypes.values
})
print(dtype_summary.to_string(index=False))


print("\n5. VALUE COUNTS PER COLUMN:")
print("=" * 80)

for col in df_merged_users.columns:
    print(f"\n{col}:")
    print(f"  Unique values: {df_merged_users[col].nunique()}")
    if df_merged_users[col].dtype in ['object', 'str', 'int64', 'float64']:
        # Untuk kolom string/object, tampilkan value counts
        value_counts = df_merged_users[col].value_counts(dropna=False)
        if len(value_counts) <= 10:
            print(value_counts)
        else:
            print(value_counts.head(10))
            print(f"  ... dan {len(value_counts) - 10} nilai lainnya")
    else:
        # Untuk kolom numerik, tampilkan statistik
        print(f"  Min: {df_merged_users[col].min()}")
        print(f"  Max: {df_merged_users[col].max()}")
        print(f"  Mean: {df_merged_users[col].mean():.2f}")

In [ ]:
# Analisis df_old['karyawan'] untuk duplicate columns, null values, dan informasi lainnya

print("="*80)
print("ANALISIS df_old['karyawan']")
print("="*80)

# 1. Info umum dataframe
print("\n1. INFORMASI UMUM:")
print(f"   Shape: {df_old['karyawan'].shape}")
print(f"   Total Rows: {len(df_old['karyawan'])}")
print(f"   Total Columns: {len(df_old['karyawan'].columns)}")

# 2. Cek duplicate columns
print("\n2. DUPLICATE COLUMNS:")
duplicate_cols = df_old['karyawan'].columns[df_old['karyawan'].columns.duplicated()]
if len(duplicate_cols) > 0:
    print(f"   ⚠️  Ditemukan {len(duplicate_cols)} duplicate column(s):")
    for col in duplicate_cols.unique():
        print(f"      - {col}")
else:
    print("   ✓ Tidak ada duplicate columns")

# 3. Cek null values per column
print("\n3. NULL VALUES PER COLUMN:")
null_counts = df_old['karyawan'].isnull().sum()
null_percentage = (null_counts / len(df_old['karyawan'])) * 100
null_info = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Null_Percentage': null_percentage.values
}).sort_values('Null_Count', ascending=False)

null_info_display = null_info[null_info['Null_Count'] > 0]
if len(null_info_display) > 0:
    print(null_info_display.to_string(index=False))
else:
    print("   ✓ Tidak ada null values")

# 4. Data types
print("\n4. DATA TYPES:")
# Display data types with a more detailed summary
dtype_summary = pd.DataFrame({
    'Column': df_old['karyawan'].dtypes.index,
    'Data_Type': df_old['karyawan'].dtypes.values
})
print(dtype_summary.to_string(index=False))


print("\n5. VALUE COUNTS PER COLUMN:")
print("=" * 80)

for col in df_old['karyawan'].columns:
    print(f"\n{col}:")
    print(f"  Unique values: {df_old['karyawan'][col].nunique()}")
    if df_old['karyawan'][col].dtype in ['object', 'str', 'int64', 'float64']:
        # Untuk kolom string/object, tampilkan value counts
        value_counts = df_old['karyawan'][col].value_counts(dropna=False)
        if len(value_counts) <= 10:
            print(value_counts)
        else:
            print(value_counts.head(10))
            print(f"  ... dan {len(value_counts) - 10} nilai lainnya")
    else:
        # Untuk kolom numerik, tampilkan statistik
        print(f"  Min: {df_old['karyawan'][col].min()}")
        print(f"  Max: {df_old['karyawan'][col].max()}")
        print(f"  Mean: {df_old['karyawan'][col].mean():.2f}")

In [ ]:
# Define a dictionary mapping the provided options to their corresponding values
bpjskerja_mapping = {
    '21034278636': '21034278636',
    '21034278651': '21034278651',
    '18088838356': '18088838356',
    '21034278685': '21034278685',
    '17008686762': '17008686762',
    '21034278669': '21034278669',
    '21056548478': '21056548478',
    '23182511560': '23182511560',
    '35782566048': '35782566048',
    '17008687224': '17008687224',
    '23074206378': '23074206378',
    '17008685640': '17008685640',
    '21012951303': '21012951303',
    '23153213873': '23153213873'
}

# Create a new column 'bpjskerja_mapped' with the mapped values
df_old['karyawan']['bpjskerja'] = df_old['karyawan']['bpjskerja'].map(bpjskerja_mapping)

# Replace missing values with None
df_old['karyawan']['bpjskerja'].fillna(value=None, inplace=True)

# Verifikasi hasil
print(df_old['karyawan']['bpjskerja'].value_counts())

In [ ]:
# Define a dictionary mapping the provided options to their corresponding values
anakke_mapping = {
    '1': 1,
    '2': 2,
    'NaN': None,
    '3': 3,
    '-': None,
    'satu': 1,
    '4': 4,
    'Pertama': 1
}

# Create a new column 'anakke' in df_old['karyawan'] with the provided options
df_old['karyawan']['anakke'] = df_old['karyawan']['anakke'].replace(anakke_mapping)

# Verifikasi hasil
print(df_old['karyawan']['anakke'].value_counts())

In [ ]:
# Define a dictionary mapping the provided options to their corresponding values
warga_mapping = {
    'Indonesia': 'WNI',
    'NaN': None,
    'WNI': 'WNI',
    'INDONESIA': 'WNI',
    '-': None,
    'fghn': None,
    'Surabaya': 'WNI'
}

# Create a new column 'warga' in df_old['karyawan'] with the provided options
df_old['karyawan']['warga'] = df_old['karyawan']['warga'].replace(warga_mapping)

# Verifikasi hasil
print(df_old['karyawan']['warga'].value_counts())

In [ ]:
# Define a dictionary mapping the provided options to their corresponding values
goldar_mapping = {
    'O': 'O',
    'B': 'B',
    '-': None,
    'AB': 'AB',
    'Ti': None,
    'O+': 'O+',
    '0': None,
    'A': 'A',
    'fg': None
}

# Create a new column 'goldar' in df_old['karyawan'] with the provided options
df_old['karyawan']['goldar'] = df_old['karyawan']['goldar'].replace(goldar_mapping)

# Verifikasi hasil
print(df_old['karyawan']['goldar'].value_counts())

In [ ]:
# Define a dictionary mapping old values to new values
moda_mapping = {
    'Motor Pribadi':  'Sepeda Motor',
    'Motor pribadi':  'Sepeda Motor',
    'Motor':  'Sepeda Motor',
    '-': None,
    'SEPEDA MOTOR': 'Sepeda Motor',
    'Sepeda Motor Pribadi': 'Sepeda Motor',
    'ghj': None,
    'Sepeda motor': 'Sepeda Motor',
    'Mobil': 'Mobil',
    'sepeda motor': 'Sepeda Motor',
    'Transportasi umum': 'Transportasi Umum',
    'Personal Motor':  'Sepeda Motor',
    'Motor Pribadi':  'Sepeda Motor',
    'Jalan Kaki': 'Jalan Kaki',
    'grab / mobil pribadi': 'Grab / Mobil Pribadi',
    'Sepeda kayuh': 'Sepeda Kayuh',
    'Grab Bike / Car': 'Grab',
    'Tidak ada': None
}

# Use replace() to fix the values in 'moda' column
df_old['karyawan']['moda'] = df_old['karyawan']['moda'].replace(moda_mapping)

# Verifikasi hasil
print(df_old['karyawan']['moda'].value_counts())

In [ ]:
# Menggunakan replace()
df_old['karyawan']['jk'] = df_old['karyawan']['jk'].replace({
    'Wanita': 'Perempuan',
    'Pria': 'Laki-laki',
    'Laki - Laki': 'Laki-laki'
})

# Verifikasi hasil
print(df_old['karyawan']['jk'].value_counts())

In [ ]:
# Standardize agama (religion) values
agama_mapping = {
    'Islam': 'Islam',
    'ISLAM': 'Islam',
    'Kristen': 'Kristen Protestan',
    'Katholik': 'Katolik',
    'Katolik': 'Katolik',
    'Hindu': 'Hindu',
    'djhxt': None,
    'Buddha': 'Buddha',
    'Konghucu': 'Konghucu',
    '-': None,  # Convert '-' to NaN
}

df_old['karyawan']['agama'] = df_old['karyawan']['agama'].replace(agama_mapping)

# Map any remaining values to None (NaN)
df_old['karyawan']['agama'] = df_old['karyawan']['agama'].apply(
    lambda x: x if x in agama_mapping.values() else None
)

df_old['karyawan']['agama'].value_counts()

In [ ]:
# Standardize status pernikahan (marital status) values
status_mapping = {
    'Menikah': 'Menikah',
    'KAWIN': 'Menikah',
    'Kawin': 'Menikah',
    'Belum Menikah': 'Belum Menikah',
    'Belum menikah': 'Belum Menikah',
    'BELUM KAWIN': 'Belum Menikah',
    'Belum Kawin': 'Belum Menikah',
    'Belum kawin': 'Belum Menikah',
    'Single ': 'Belum Menikah',
    'Lajang': 'Belum Menikah',
    'Belum': 'Belum Menikah',
    'Belum nikah ': 'Belum Menikah',
    '-': None,
    'dxhj': None,
}

df_old['karyawan']['status'] = df_old['karyawan']['status'].replace(status_mapping)

# Verify results
df_old['karyawan']['status'].value_counts()

In [ ]:
# Standardize status values (Aktif -> 1, Non Aktif -> 0)
df_merged_users['status'] = df_merged_users['status'].replace({
    'Aktif': 1,
    'Non Aktif': 0
})

# Verify results
df_merged_users['status'].value_counts()

In [ ]:
# # 5. Display semua baris dan kolom
# print("\n5. TAMPILAN SEMUA DATA:")
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', None)
# pd.set_option('display.max_colwidth', None)
# display(df_merged_users)

In [ ]:
df_new['karyawan'].info()

In [ ]:
df_new['karyawan'].info()

In [ ]:
# df_new['karyawan']['id_karyawan'] = df_old['karyawan']['idkaryawan']
# df_new['karyawan']['id_user'] = df_old['karyawan']['idusers']
# df_new['karyawan']['nik_ktp'] = df_old['karyawan']['ktp']
# df_new['karyawan']['nama_lengkap'] = df_old['karyawan']['nama']
# df_new['karyawan']['nama_panggilan'] = df_old['karyawan']['nickname']
# df_new['karyawan']['tempat_lahir'] = df_old['karyawan']['kota']
# df_new['karyawan']['tanggal_lahir'] = df_old['karyawan']['tgl']
# df_new['karyawan']['jenis_kelamin'] = df_old['karyawan']['jk']
# df_new['karyawan']['golongan_darah'] = df_old['karyawan']['goldar']
# df_new['karyawan']['agama'] = df_old['karyawan']['agama']
# df_new['karyawan']['status_pernikahan'] = df_old['karyawan']['status']
# df_new['karyawan']['alamat_ktp'] = df_old['karyawan']['alamatktp']
# df_new['karyawan']['alamat_domisili'] = df_old['karyawan']['domisili']
# df_new['karyawan']['kewarganegaraan'] = df_old['karyawan']['warga']
# df_new['karyawan']['anak_ke'] = df_old['karyawan']['anakke']
# df_new['karyawan']['jumlah_anak'] = df_old['karyawan']['anak']
# df_new['karyawan']['hobi'] = df_old['karyawan']['hobi']
# df_new['karyawan']['akun_linkedin'] = df_old['karyawan']['linkedin']
# df_new['karyawan']['email_pribadi'] = df_old['karyawan']['email']
# df_new['karyawan']['email_kantor'] = df_old['karyawan']['emailkantor']
# df_new['karyawan']['nomor_telepon'] = df_old['karyawan']['telp']
# df_new['karyawan']['nomor_npwp'] = df_old['karyawan']['npwp']
# df_new['karyawan']['bpjs_ketenagakerjaan'] = df_old['karyawan']['bpjskerja']
# df_new['karyawan']['bpjs_kesehatan'] = df_old['karyawan']['bpjssehat']
# df_new['karyawan']['nomor_rekening'] = df_old['karyawan']['rekening']
# df_new['karyawan']['moda_transportasi'] = df_old['karyawan']['moda']
# df_new['karyawan']['akun_instagram'] = df_old['karyawan']['ig']
# df_new['karyawan']['akun_facebook'] = df_old['karyawan']['fb']
# df_new['karyawan']['link_dokumen_pribadi'] = df_old['karyawan']['link']
# df_new['karyawan']['riwayat_kesehatan'] = df_old['karyawan']['riwayat']
# df_new['karyawan']['tahun_mulai_kerja'] = df_old['users']['thnbekerja']
# df_new['karyawan']['keahlian'] = df_old['users']['expertise']
# df_new['karyawan']['id_shift'] = df_old['users']['idjamkerja']
# df_new['karyawan']['status_aktif'] = df_old['users']['status']
# df_new['karyawan']['foto_profile'] = df_old['users']['foto']
# df_new['karyawan']['ttd_digital'] = df_old['users']['ttd']

## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection